# SCUC Paper Suite Notebook

This notebook runs the paper-suite presets for `scuc`.

Workflow:
1. Pick one config from `experiments/scuc/configs/paper_suite`.
2. Run `dry-run` to verify.
3. Run the real command for that config.
4. Optionally run all paper-suite configs in batch.


In [ ]:
using Pkg

function find_repo_root(start::AbstractString = pwd())
    dir = abspath(start)
    while true
        has_project = isfile(joinpath(dir, "Project.toml"))
        has_entry = isfile(joinpath(dir, "experiments/scuc/run_experiment.jl"))
        if has_project && has_entry
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate repository root from $(start).")
        dir = parent
    end
end

repo_root = find_repo_root()
cd(repo_root)
Pkg.activate(repo_root)

run_script = joinpath("experiments/scuc/run_experiment.jl")
paper_suite_dir = joinpath("experiments/scuc/configs/paper_suite")

configs = sort(filter(f -> endswith(f, ".jl"), readdir(joinpath(repo_root, paper_suite_dir))))
println("Repo root: ", repo_root)
println("Run script: ", run_script)
println("Paper suite configs:")
for (i, f) in enumerate(configs)
    println(lpad(i, 2), ". ", f)
end


In [ ]:
config_path = joinpath(paper_suite_dir, "exp4_idbc_inherit_on_bestnorm.jl")

# Run the real command for that config
cd(repo_root) do
    run(`julia --project=. $run_script $config_path`)
end

In [ ]:
# Select one config by name (edit this line):
selected_config = "exp1_idbc_dynamic_inherit_on_snc.jl"

# Backward-compatible aliases for old paper-suite filenames.
legacy_aliases = Dict(
    "exp1_dbc_dynamic_inherit_off_alpha.jl" => "exp1_idbc_dynamic_inherit_off_alpha.jl",
    "exp1_dbc_dynamic_inherit_off_regular.jl" => "exp1_idbc_dynamic_inherit_off_regular.jl",
    "exp1_dbc_dynamic_inherit_off_snc.jl" => "exp1_idbc_dynamic_inherit_off_snc.jl",
    "exp1_dbc_dynamic_inherit_on_alpha.jl" => "exp1_idbc_dynamic_inherit_on_alpha.jl",
    "exp1_dbc_dynamic_inherit_on_regular.jl" => "exp1_idbc_dynamic_inherit_on_regular.jl",
    "exp1_dbc_dynamic_inherit_on_snc.jl" => "exp1_idbc_dynamic_inherit_on_snc.jl",
    "exp2_dbc_fixedD10_inherit_on.jl" => "exp2_idbc_fixedD10_inherit_on.jl",
    "exp4_idbc_inherit_on_bestnorm.jl" => "exp4_dbc_inherit_on_bestnorm.jl",
)

resolved_config = get(legacy_aliases, selected_config, selected_config)
if resolved_config != selected_config
    println("Mapped legacy config `", selected_config, "` -> `", resolved_config, "`")
end

config_path = joinpath(paper_suite_dir, resolved_config)
full_config_path = joinpath(repo_root, config_path)
if !isfile(full_config_path)
    println("Available configs in paper_suite:")
    for f in configs
        println("  - ", f)
    end
    error("Config not found: $(full_config_path)")
end

println("Selected config: ", config_path)


In [ ]:
# Dry-run (recommended first)
cd(repo_root) do
    run(`julia --project=. $run_script $config_path --dry-run`)
end


In [ ]:
# Real run for the selected config
cd(repo_root) do
    run(`julia --project=. $run_script $config_path`)
end

In [ ]:
# Optional: run all paper-suite configs (this can take a long time)
cd(repo_root) do
    for cfg in configs
        cfg_path = joinpath(paper_suite_dir, cfg)
        println("\n=== Running: ", cfg_path, " ===")
        run(`julia --project=. $run_script $cfg_path`)
    end
end
